In [3]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch, transformers, trl, peft, pymatgen.core
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| trl", trl.__version__, "| peft", peft.__version__)

print("MPS available:", torch.backends.mps.is_available())
x = torch.ones(3, device="mps")
print("Test computation on GPU:", (x * 2).tolist())

torch 2.14.0 | transformers 5.17.0 | trl 1.13.0 | peft 0.21.0
MPS available: True
Test computation on GPU: [2.0, 2.0, 2.0]


In [4]:
# Phase 1, Cell 1: Pull the final chemical formula out of a model response.
# Handles markdown formatting (**Formula:**) and subscript digits (Li₂MnO₃).
import re

SUBSCRIPTS = str.maketrans("₀₁₂₃₄₅₆₇₈₉−", "0123456789-")

def normalize(text):
    return text.translate(SUBSCRIPTS)

def extract_formula(text):
    text = normalize(text)
    matches = re.findall(r"[Ff]ormula:[\s*`]*([A-Z][A-Za-z0-9()]*)", text)
    return matches[-1] if matches else None

In [5]:
# Phase 1, Cell 2: The reward function. Scores a response as -1 (unusable),
# 0 (valid formula, not charge-balanced), or 1 (charge-balanced, correct elements).
from pymatgen.core import Composition

def score_response(text, required_elements):
    """Return -1 if unusable, 0 if valid but not charge-balanced, 1 if balanced."""
    formula = extract_formula(text)
    if formula is None:
        return -1
    try:
        comp = Composition(formula)
    except Exception:
        return -1
    if {el.symbol for el in comp.elements} != set(required_elements):
        return -1
    return 1 if comp.oxi_state_guesses() else 0

In [6]:
# Phase 1, Cell 3: Check whether the oxidation states the model stated in its
# reasoning actually sum to zero for the formula it gave. Used for analysis, not reward.
def stated_states_match(text):
    text = normalize(text)
    formula = extract_formula(text)
    states = dict(re.findall(r"\b([A-Z][a-z]?):\s*([+-]?\d+)", text))
    if formula is None:
        return False
    try:
        comp = Composition(formula)
    except Exception:
        return False
    symbols = {el.symbol for el in comp.elements}
    if not symbols <= set(states):
        return False
    total = sum(comp[s] * int(states[s]) for s in symbols)
    return abs(total) < 1e-6

In [7]:
# Phase 1, Cell 4: Test the scorer and faithfulness check on cases with known answers.
# All eight lines should print PASS before moving on to Phase 2.
GOOD = "Li: +1, Mn: +4, O: -2\nCheck: 2(+1) + (+4) + 3(-2) = 0\nFormula: Li2MnO3"
REQ = ["Li", "Mn", "O"]

tests = [
    ("balanced",             score_response(GOOD, REQ),                          1),
    ("markdown bold",        score_response("**Formula:** Li2MnO3", REQ),        1),
    ("unicode subscripts",   score_response("Formula: Li₂MnO₃", REQ),            1),
    ("unbalanced, Mn +9",    score_response("Formula: LiMnO5", REQ),             0),
    ("wrong elements",       score_response("Formula: H2O", REQ),               -1),
    ("no formula line",      score_response("Li2MnO3 is nice", REQ),            -1),
    ("faithful reasoning",   stated_states_match(GOOD),                          True),
    ("unfaithful reasoning", stated_states_match(GOOD.replace("Mn: +4", "Mn: +2")), False),
]
for name, got, expected in tests:
    print("PASS" if got == expected else "FAIL", "|", name, "| expected", expected, "got", got)

PASS | balanced | expected 1 got 1
PASS | markdown bold | expected 1 got 1
PASS | unicode subscripts | expected 1 got 1
PASS | unbalanced, Mn +9 | expected 0 got 0
PASS | wrong elements | expected -1 got -1
PASS | no formula line | expected -1 got -1
PASS | faithful reasoning | expected True got True
PASS | unfaithful reasoning | expected False got False


In [8]:
# Phase 2, Cell 5: Define the element pools and prompt template, then form
# every combination of one cation, one transition metal, and one anion (220 total).
import itertools, json, os, random
from pymatgen.core import Element

CATIONS = ["Li", "Na", "K", "Mg", "Ca"]
METALS  = ["Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn", "Nb", "Mo"]
ANIONS  = ["O", "S", "F", "N"]

TEMPLATE = (
    "Propose a plausible inorganic compound containing {a}, {b}, and {c}.\n"
    "First state the oxidation state you will use for each element.\n"
    "Then give the compound on a final line in the form 'Formula: <formula>'."
)

def name(symbol):
    return Element(symbol).long_name.lower()

combos = list(itertools.product(CATIONS, METALS, ANIONS))
print(len(combos), "element combinations")

220 element combinations


In [9]:
# Phase 2, Cell 6: Shuffle with a fixed seed, hold out one fifth for testing,
# and save both splits to the data/ folder so every later phase uses the same prompts.
random.seed(0)
random.shuffle(combos)
n_test = len(combos) // 5
test_combos, train_combos = combos[:n_test], combos[n_test:]

def to_record(combo):
    a, b, c = combo
    return {"elements": list(combo), "prompt": TEMPLATE.format(a=name(a), b=name(b), c=name(c))}

os.makedirs("data", exist_ok=True)
for split, items in [("train", train_combos), ("test", test_combos)]:
    with open(f"data/{split}_prompts.json", "w") as f:
        json.dump([to_record(c) for c in items], f, indent=2)

print(len(train_combos), "train |", len(test_combos), "test")
print(to_record(test_combos[0])["prompt"])

176 train | 44 test
Propose a plausible inorganic compound containing calcium, copper, and fluorine.
First state the oxidation state you will use for each element.
Then give the compound on a final line in the form 'Formula: <formula>'.


In [12]:
# Phase 3, Cell 7: Load the 1.5B model and tokenizer from the local cache onto the Apple GPU.
# The files were downloaded separately with `hf download`; local_files_only avoids any network access.
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

DEVICE = "mps"
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32,
                                             local_files_only=True).to(DEVICE)
model.eval()
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters on {model.device}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

1544M parameters on mps:0


In [13]:
# Phase 3, Cell 8: Generate several responses per prompt, in batches.
# Each prompt is preceded by one worked example so the model sees the exact answer format.
EXAMPLE_Q = TEMPLATE.format(a="strontium", b="aluminum", c="oxygen")
EXAMPLE_A = "Sr: +2, Al: +3, O: -2\nCheck: (+2) + 2(+3) + 4(-2) = 0\nFormula: SrAl2O4"

def build_input(prompt):
    messages = [{"role": "user", "content": EXAMPLE_Q},
                {"role": "assistant", "content": EXAMPLE_A},
                {"role": "user", "content": prompt}]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def generate(prompts, n_samples=8, batch_prompts=4, max_new_tokens=150):
    results = []
    for i in range(0, len(prompts), batch_prompts):
        batch = [build_input(p) for p in prompts[i:i + batch_prompts]]
        inputs = tok(batch, return_tensors="pt", padding=True).to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, do_sample=True, num_return_sequences=n_samples,
                                 max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id)
        texts = tok.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        results += [texts[j:j + n_samples] for j in range(0, len(texts), n_samples)]
        print(f"{min(i + batch_prompts, len(prompts))}/{len(prompts)} prompts done")
    return results

print(model.generation_config)

GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.1,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}



In [14]:
# Phase 3, Cell 9 (revised): Check the format on three prompts, four responses each.
# A broader sample than before, to judge whether the fix works before the full run.
set_seed(0)
test = json.load(open("data/test_prompts.json"))
groups = generate([t["prompt"] for t in test[:3]], n_samples=4)
for t, group in zip(test[:3], groups):
    for r in group:
        print(r, "\n-> score:", score_response(r, t["elements"]), "\n" + "-" * 40)

3/3 prompts done
Ca: +2, Cu: +1, F: -1
Check: (+2) + (+1) + (-1) x 5 = 0
Formula: CaCuF5 
-> score: 1 
----------------------------------------
Ca: +2, Cu: +2, F: -1
Check: (+2) + 2(+2) + (-1) = 0
Formula: CaCuF2 
-> score: 0 
----------------------------------------
Ca: +2, Cu: +1, F: -1
Check: (+2) + (+1) + (-1) = 0
Formula: CaCuF2 
-> score: 0 
----------------------------------------
Ca: +2, Cu: +1, F: -1
Check: (+2) + (+1) + 4(-1) = 0
Formula: CaCuF2 
-> score: 0 
----------------------------------------
Na: +1, Mn: +7, F: -1
Check: (+1) + 7(+7) + (-1)(4) = 0
Formula: NaMnF4 
-> score: 1 
----------------------------------------
Na: +1, Mn: +7, F: -1
Check: (+1) + 7(+7) + (-1)(4) = 0
Formula: NaMF4 
-> score: -1 
----------------------------------------
Na: +1, Mn: +7, F: -1
Check: (+1) + 7(+7) + 5(-1) = 0
Formula: NaMnF5 
-> score: 1 
----------------------------------------
Na: +1, Mn: +7, F: -1
Check: (+1) + 7(+7) + (-1)(4) = 0
Formula: NaMnF4 
-> score: 1 
--------------------

In [ ]:
#Qwen 0.5B systematically omitted the transition metal, collapsing to familiar binary compounds. This could be due to 
#lack of TMO_x in training data, so instead using 1.5B

#interestingly, the 1.5B model seems to output a correct compound but fails toprovide the right oxidation reasoning. so the
#experiment here becomes will the dpo actually make the model's oxidation reasoning better (arguably more important) or just
#raise formula accuracy merely.

In [17]:
# Phase 3, Cell 10: Generate 8 responses for each of the 44 held-out prompts (352 total)
# and save them, so the metrics can be recomputed or inspected later without regenerating.
set_seed(0)
base_responses = generate([t["prompt"] for t in test])
with open("data/baseline_responses.json", "w") as f:
    json.dump([{"elements": t["elements"], "responses": r} for t, r in zip(test, base_responses)],
              f, indent=2)

4/44 prompts done
8/44 prompts done
12/44 prompts done
16/44 prompts done
20/44 prompts done
24/44 prompts done
28/44 prompts done
32/44 prompts done
36/44 prompts done
40/44 prompts done
44/44 prompts done


In [18]:
# Phase 3, Cell 11: Compute validity, balance, diversity, and faithfulness from the saved responses.
# These are the baseline numbers the trained model will be compared against in Phase 6.
def metrics(records):
    scores, faithful, diversity = [], [], []
    for rec in records:
        formulas = [extract_formula(r) for r in rec["responses"]]
        diversity.append(len({f for f in formulas if f}) / len(formulas))
        for r in rec["responses"]:
            s = score_response(r, rec["elements"])
            scores.append(s)
            if s == 1:
                faithful.append(stated_states_match(r))
    n = len(scores)
    return {"validity": sum(s >= 0 for s in scores) / n,
            "balance": sum(s == 1 for s in scores) / n,
            "diversity": sum(diversity) / len(diversity),
            "faithfulness": sum(faithful) / max(len(faithful), 1),
            "n_responses": n}

with open("data/baseline_responses.json") as f:
    baseline = metrics(json.load(f))
with open("data/baseline_metrics.json", "w") as f:
    json.dump(baseline, f, indent=2)
print(baseline)

{'validity': 0.8465909090909091, 'balance': 0.5596590909090909, 'diversity': 0.3465909090909091, 'faithfulness': 0.3147208121827411, 'n_responses': 352}


In [19]:
# Phase 3, Cell 12: Print five balanced responses that the faithfulness check marked unfaithful.
# Confirms the low faithfulness rate reflects real reasoning errors, not a parsing problem.
with open("data/baseline_responses.json") as f:
    records = json.load(f)
shown = 0
for rec in records:
    for r in rec["responses"]:
        if shown < 5 and score_response(r, rec["elements"]) == 1 and not stated_states_match(r):
            print(r, "\n" + "-" * 40)
            shown += 1

Ca: +2, Cu: +1, F: -1
Check: (+2) + (+1) + (-1) × 5 = 0
Formula: CaCuF5 
----------------------------------------
Ca: +2, Cu: +1, F: -1
Check: (+2) + (+1) + (-1)(4) = 0
Formula: CaCuF4 
----------------------------------------
Na: +1, Mn: +7, F: -1
Check: (+1) + 7(+7) + (-1)(4) = 0
Formula: NaMnF4 
----------------------------------------
Na: +1, Mn: +7, F: -1
Check: (+1) + 7(+7) + (-1)(4) = 0
Formula: NaMnF4 
----------------------------------------
Na: +1, Mn: +7, F: -1
Check: (+1) + 7(+7) + 4(-1) = 0
Formula: Na2MnF4 
----------------------------------------
